# Digital Communication Laboratory
## Experiment 9: Amplitude Shift Keying (ASK) and Frequency Shift Keying (FSK)

**Programme:** B.Tech in Electronics and Communication Engineering  
**Platform:** Python 3 (NumPy, SciPy, Matplotlib, Pandas)  
**Mode:** Self-Contained First-Principles Simulation without Proprietary Toolboxes  

---

### Objectives & Syllabus Coverage:
1. **Passband Waveform Synthesis:** Synthesize binary ASK (OOK) and continuous-phase binary FSK (CPFSK) waveforms.
2. **Power Spectral Density Analysis:** Compute and compare Welch PSDs of BASK and BFSK, illustrating bandwidth differences.
3. **Mandatory Syllabus Validation:** Numerically evaluate basis inner products $\langle \phi_1, \phi_2 \rangle$ over $[0, T_b]$ across multiple tone spacings to prove orthogonality conditions ($\Delta f = 1/(2T_b)$ coherent, $\Delta f = 1/T_b$ non-coherent).
4. **BFSK Tone Spacing Orthogonality Sweep:** Sweep $\Delta f$ continuously from $0$ to $3 R_b$ to map cross-correlation nulls.
5. **Coherent Correlator Demodulation:** Construct matched correlator receivers for ASK and BFSK, illustrating threshold slicing and differential decision rules.
6. **Decision Statistic Distributions:** Plot empirical histograms/PDFs of decision variables under AWGN to visualize decision regions and threshold optimality.
7. **Non-Coherent Envelope Detection:** Implement Hilbert envelope extraction followed by low-pass filtering and slicing.
8. **Monte Carlo BER Performance Comparison:** Simulate BER over $E_b/N_0 \in [0, 14]\text{ dB}$ for all 4 schemes and compare against exact theoretical curves.


In [ ]:
# Cell 1: Environment Setup, Library Imports, and Publication Styling
%matplotlib inline
import os
import sys
from pathlib import Path
from typing import Tuple, Dict, List, Optional

import numpy as np
import scipy.signal as signal
import scipy.special as sp
import matplotlib.pyplot as plt
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

# Set plot styling for publication-quality visuals
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "lines.linewidth": 1.5,
    "grid.alpha": 0.5,
    "grid.linestyle": "--",
    "savefig.dpi": 300,
    "savefig.bbox": "tight"
})

# Core System Parameters (Strict Experiment Specifications)
RB = 1000.0                # Bit rate = 1000 bits/sec
TB = 1.0 / RB              # Bit duration = 1.0 ms
FS = 100000.0              # Sampling frequency = 100 kHz
SPS = int(FS * TB)         # Samples per bit = 100 samples
FC_ASK = 10000.0           # ASK Carrier frequency = 10 kHz (10 cycles per bit)
F1_FSK = 10000.0           # BFSK Mark Frequency ('1') = 10 kHz
F2_FSK = 12000.0           # BFSK Space Frequency ('0') = 12 kHz
AMP = 1.0                  # Transmit peak carrier amplitude (Volts)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("[OK] Environment initialized. Parameters locked:")
print(f"    Rb = {RB:.0f} bps | Tb = {TB*1000:.1f} ms | fs = {FS/1000:.0f} kHz | sps = {SPS}")
print(f"    ASK fc = {FC_ASK/1000:.0f} kHz | BFSK f1 = {F1_FSK/1000:.0f} kHz, f2 = {F2_FSK/1000:.0f} kHz (Delta_f = 2 kHz)")
print(f"    Outputs directory: {OUTPUT_DIR.resolve()}")


---
## 1. Signal Generation: Passband ASK and BFSK Waveforms

### 1.1 Binary ASK (On-Off Keying)
$$s_{\text{ASK}}(t) = \begin{cases} A \cos(2\pi f_c t), & \text{bit } 1 \\ 0, & \text{bit } 0 \end{cases}$$

### 1.2 Binary FSK (Continuous-Phase FSK)
$$s_{\text{FSK}}(t) = \begin{cases} A \cos(2\pi f_1 t + \theta_i), & \text{bit } 1 \\ A \cos(2\pi f_2 t + \theta_i), & \text{bit } 0 \end{cases}$$
Continuous phase ensures smooth transitions at bit boundaries, suppressing high-frequency spectral side lobes.


In [ ]:
# Cell 2: Passband Waveform Generation [Figure 1]
def generate_random_bits(n_bits: int, seed: Optional[int] = 42) -> np.ndarray:
    rng = np.random.default_rng(seed)
    return rng.integers(0, 2, size=n_bits)

def modulate_ask(bits: np.ndarray, tb: float = TB, fc: float = FC_ASK, fs: float = FS, amp: float = AMP):
    n_bits = len(bits)
    sps = int(fs * tb)
    total_samples = n_bits * sps
    t = np.arange(total_samples) / fs
    baseband = np.repeat(bits, sps)
    carrier = amp * np.cos(2.0 * np.pi * fc * t)
    ask_signal = baseband * carrier
    return t, baseband, ask_signal

def modulate_bfsk(bits: np.ndarray, tb: float = TB, f1: float = F1_FSK, f2: float = F2_FSK, fs: float = FS, amp: float = AMP, phase_continuous: bool = True):
    n_bits = len(bits)
    sps = int(fs * tb)
    total_samples = n_bits * sps
    t = np.arange(total_samples) / fs
    baseband = np.repeat(bits, sps)
    bfsk_signal = np.zeros(total_samples)
    current_phase = 0.0
    for i, b in enumerate(bits):
        idx_start = i * sps
        idx_end = (i + 1) * sps
        t_segment = t[idx_start:idx_end] - t[idx_start]
        freq = f1 if b == 1 else f2
        if phase_continuous:
            phase = current_phase + 2.0 * np.pi * freq * t_segment
            bfsk_signal[idx_start:idx_end] = amp * np.cos(phase)
            current_phase = (current_phase + 2.0 * np.pi * freq * tb) % (2.0 * np.pi)
        else:
            bfsk_signal[idx_start:idx_end] = amp * np.cos(2.0 * np.pi * freq * t[idx_start:idx_end])
    return t, baseband, bfsk_signal

# Generate demo sequence and plot Figure 1
demo_bits = np.array([1, 0, 1, 1, 0, 1, 0, 0, 1, 0])
t, bb, ask_sig = modulate_ask(demo_bits, TB, FC_ASK, FS, AMP)
_, _, bfsk_sig = modulate_bfsk(demo_bits, TB, F1_FSK, F2_FSK, FS, AMP, phase_continuous=True)

fig, axes = plt.subplots(4, 1, figsize=(13, 9), sharex=True)
axes[0].step(t * 1000, bb, where="post", color="navy", linewidth=2.0)
axes[0].set_ylabel("Data Bits"); axes[0].set_ylim(-0.2, 1.2)
axes[0].set_title(f"Binary Input Sequence: {list(demo_bits)}")
for i, b in enumerate(demo_bits):
    axes[0].text((i + 0.5) * TB * 1000, 0.5, f"b={b}", ha="center", va="center", fontsize=11, fontweight="bold", color="darkred")

carrier = AMP * np.cos(2.0 * np.pi * FC_ASK * t)
axes[1].plot(t * 1000, carrier, color="gray", linewidth=1.0)
axes[1].set_ylabel("Carrier"); axes[1].set_title(r"Unmodulated Carrier Reference ($f_c = 10$ kHz)")

axes[2].plot(t * 1000, ask_sig, color="crimson", linewidth=1.2)
axes[2].set_ylabel("ASK $s(t)$"); axes[2].set_title(r"Binary ASK (OOK) Passband Waveform: $s_1(t) = A\cos(2\pi f_c t), s_0(t) = 0$")

axes[3].plot(t * 1000, bfsk_sig, color="forestgreen", linewidth=1.2)
axes[3].set_ylabel("BFSK $s(t)$"); axes[3].set_xlabel("Time (milliseconds)")
axes[3].set_title(r"Binary FSK (CPFSK) Waveform: Mark $f_1 = 10$ kHz, Space $f_2 = 12$ kHz")

for ax in axes:
    ax.grid(True)
    for i in range(len(demo_bits) + 1):
        ax.axvline(i * TB * 1000, color="black", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "1_ask_bfsk_passband_waveforms.png")
plt.show()


---
## 2. Power Spectral Density (PSD) Analysis

Using Welch's averaged modified periodogram method, we compute the Power Spectral Densities of BASK and BFSK.
- **BASK Transmission Bandwidth:** $B_{\text{ASK}} = 2 R_b = 2000\text{ Hz}$ (first null-to-null bandwidth centered at $f_c$).
- **BFSK Transmission Bandwidth:** $B_{\text{FSK}} \approx |f_1 - f_2| + 2 R_b = 2000 + 2000 = 4000\text{ Hz}$ (Carson's rule).


In [ ]:
# Cell 3: Power Spectral Density Estimation via Welch's Method [Figure 2]
psd_bits = generate_random_bits(5000, seed=555)
_, _, ask_psd_sig = modulate_ask(psd_bits, TB, FC_ASK, FS, AMP)
_, _, bfsk_psd_sig = modulate_bfsk(psd_bits, TB, F1_FSK, F2_FSK, FS, AMP, phase_continuous=True)

nperseg = 4096
f_ask, psd_ask = signal.welch(ask_psd_sig, FS, nperseg=nperseg, window="hann")
f_bfsk, psd_bfsk = signal.welch(bfsk_psd_sig, FS, nperseg=nperseg, window="hann")
psd_ask_db = 10.0 * np.log10(psd_ask / np.max(psd_ask) + 1e-12)
psd_bfsk_db = 10.0 * np.log10(psd_bfsk / np.max(psd_bfsk) + 1e-12)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax1.plot(f_ask / 1000.0, psd_ask_db, color="crimson", linewidth=1.5, label="BASK (OOK) PSD")
ax1.axvline(FC_ASK / 1000.0, color="black", linestyle="--", label=r"Carrier $f_c = 10$ kHz")
ax1.axvline((FC_ASK - RB) / 1000.0, color="gray", linestyle=":", label=r"Nulls ($f_c \pm R_b$)")
ax1.axvline((FC_ASK + RB) / 1000.0, color="gray", linestyle=":")
ax1.set_ylabel("Normalized PSD (dB/Hz)")
ax1.set_title("Power Spectral Density of Binary ASK (OOK)")
ax1.set_ylim(-60, 5); ax1.grid(True); ax1.legend(loc="upper right")

ax2.plot(f_bfsk / 1000.0, psd_bfsk_db, color="forestgreen", linewidth=1.5, label="BFSK PSD")
ax2.axvline(F1_FSK / 1000.0, color="blue", linestyle="--", label=r"Mark $f_1 = 10$ kHz")
ax2.axvline(F2_FSK / 1000.0, color="purple", linestyle="--", label=r"Space $f_2 = 12$ kHz")
ax2.set_xlabel("Frequency (kHz)"); ax2.set_ylabel("Normalized PSD (dB/Hz)")
ax2.set_title("Power Spectral Density of Binary FSK (BFSK)")
ax2.set_xlim(5, 18); ax2.set_ylim(-60, 5); ax2.grid(True); ax2.legend(loc="upper right")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "2_power_spectral_density.png")
plt.show()


---
## 3. Mandatory Syllabus Validation: BFSK Basis Orthogonality

> **Syllabus Requirement:** *"Calculate inner products of the BFSK basis signals over one bit interval."*

The basis inner product over $[0, T_b]$ is:
$$\rho = \int_0^{T_b} \phi_1(t) \phi_2(t) dt = \frac{2}{T_b} \int_0^{T_b} \cos(2\pi f_1 t) \cos(2\pi f_2 t + \phi) dt$$
- **Coherent Orthogonality ($\phi = 0$):** Requires $\Delta f = \frac{k}{2 T_b} = \frac{k R_b}{2}$. Minimum spacing $\Delta f_{\min, \text{coh}} = 500\text{ Hz}$.
- **Non-Coherent Orthogonality (Arbitrary $\phi$):** Requires $\Delta f = \frac{k}{T_b} = k R_b$. Minimum spacing $\Delta f_{\min, \text{noncoh}} = 1000\text{ Hz}$.


In [ ]:
# Cell 4: Mandatory Syllabus Validation - Basis Inner Product Verification [Figure 8]
def validate_bfsk_orthogonality(tb: float = TB, fs: float = FS, f1: float = F1_FSK) -> pd.DataFrame:
    sps = int(fs * tb)
    t_bit = np.linspace(0.0, tb, sps + 1)
    phi1 = np.sqrt(2.0 / tb) * np.cos(2.0 * np.pi * f1 * t_bit)

    test_cases = [
        ("Sub-Nyquist / Non-orthogonal", 250.0, 0.25),
        ("Minimum Coherent Orthogonal (1/(2Tb))", 500.0, 0.50),
        ("Intermediate Non-orthogonal", 750.0, 0.75),
        ("Minimum Non-coherent Orthogonal (1/Tb)", 1000.0, 1.00),
        ("Second Coherent Orthogonal (3/(2Tb))", 1500.0, 1.50),
        ("Our Lab Setting (Delta_f = 2 kHz = 2/Tb)", 2000.0, 2.00)
    ]
    records = []
    for desc, delta_f, norm_spacing in test_cases:
        f2 = f1 + delta_f
        phi2_inphase = np.sqrt(2.0 / tb) * np.cos(2.0 * np.pi * f2 * t_bit)
        inner_prod_inphase = float(np.trapezoid(phi1 * phi2_inphase, t_bit))
        phi2_quad = np.sqrt(2.0 / tb) * np.cos(2.0 * np.pi * f2 * t_bit + np.pi / 2.0)
        inner_prod_quad = float(np.trapezoid(phi1 * phi2_quad, t_bit))
        rho_envelope = np.sqrt(inner_prod_inphase ** 2 + inner_prod_quad ** 2)
        is_coh = np.isclose(inner_prod_inphase, 0.0, atol=1e-5)
        is_noncoh = np.isclose(rho_envelope, 0.0, atol=1e-5)
        records.append({
            "Description": desc,
            "Delta_f (Hz)": delta_f,
            "Delta_f * Tb": norm_spacing,
            "Inner Product (phi=0)": inner_prod_inphase,
            "Envelope Cross-Corr": rho_envelope,
            "Coherent Orthogonal": "YES" if is_coh else "NO",
            "Non-Coherent Orthogonal": "YES" if is_noncoh else "NO"
        })
    return pd.DataFrame(records)

df_ortho = validate_bfsk_orthogonality(TB, FS, F1_FSK)
display(df_ortho)

# Plot Figure 8: Bar chart & validation table summary
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
labels = [f"$\\Delta f={r['Delta_f (Hz)']:.0f}$Hz\n({r['Delta_f * Tb']:.2f}$R_b$)" for _, r in df_ortho.iterrows()]
x = np.arange(len(labels))
width = 0.35

ax1.bar(x - width/2, df_ortho["Inner Product (phi=0)"], width, label=r"In-Phase ($\phi=0$)", color="royalblue")
ax1.bar(x + width/2, df_ortho["Envelope Cross-Corr"], width, label=r"Envelope ($\max_\phi |\rho|$)", color="crimson")
ax1.axhline(0.0, color="black", linewidth=0.8)
ax1.set_xticks(x)
ax1.set_xticklabels(labels, rotation=35, ha="right", fontsize=9)
ax1.set_ylabel(r"Inner Product $\langle \phi_1, \phi_2 \rangle$")
ax1.set_title(r"Mandatory Validation: Basis Inner Products over $[0, T_b]$", fontsize=12)
ax1.grid(True, linestyle="--", alpha=0.5)
ax1.legend(loc="upper right")

ax2.axis("off")
table_data = []
for _, r in df_ortho.iterrows():
    table_data.append([
        f"{r['Delta_f (Hz)']:.0f} Hz",
        f"{r['Delta_f * Tb']:.2f}",
        f"{r['Inner Product (phi=0)']:+.4f}",
        f"{r['Envelope Cross-Corr']:.4f}",
        r['Coherent Orthogonal'],
        r['Non-Coherent Orthogonal']
    ])
col_labels = ["Delta_f", "Norm Spacing", "Inner Prod (phi=0)", "Envelope", "Coh Ortho?", "Non-Coh Ortho?"]
table = ax2.table(cellText=table_data, colLabels=col_labels, loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.1, 1.8)
ax2.set_title("Numerical Inner Product Verification Summary", fontsize=12, pad=15)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "8_mandatory_orthogonality_validation.png")
plt.show()


---
### 3.2 BFSK Tone Spacing Orthogonality Sweep

Continuous frequency sweep of $\Delta f$ from $0$ to $3000\text{ Hz}$ ($0$ to $3 R_b$) illustrating:
- The **Coherent Orthogonality nulls** at $\Delta f = 0.5 R_b, 1.0 R_b, 1.5 R_b, \dots$ where $\rho(\Delta f, \phi=0) = 0$.
- The **Non-Coherent Orthogonality nulls** at $\Delta f = 1.0 R_b, 2.0 R_b, \dots$ where the entire envelope cross-correlation vanishes identically.


In [ ]:
# Cell 5: Continuous BFSK Tone Spacing Sweep [Figure 5]
fs_high = 200000.0
sps_high = int(fs_high * TB)
dt_high = 1.0 / fs_high
t_bit_high = np.arange(sps_high) * dt_high
f1 = 10000.0

delta_f_array = np.linspace(0.0, 3000.0, 600)
norm_spacing = delta_f_array * TB

rho_inphase = np.zeros_like(delta_f_array)
rho_quad = np.zeros_like(delta_f_array)
rho_envelope = np.zeros_like(delta_f_array)

phi1 = np.sqrt(2.0 / TB) * np.cos(2.0 * np.pi * f1 * t_bit_high)

for i, df in enumerate(delta_f_array):
    f2 = f1 + df
    phi2_i = np.sqrt(2.0 / TB) * np.cos(2.0 * np.pi * f2 * t_bit_high)
    phi2_q = np.sqrt(2.0 / TB) * np.cos(2.0 * np.pi * f2 * t_bit_high + np.pi / 2.0)
    rho_i = np.sum(phi1 * phi2_i) * dt_high
    rho_q = np.sum(phi1 * phi2_q) * dt_high
    rho_inphase[i] = rho_i
    rho_quad[i] = rho_q
    rho_envelope[i] = np.sqrt(rho_i ** 2 + rho_q ** 2)

fig, ax = plt.subplots(figsize=(12, 6.5))
ax.plot(norm_spacing, rho_inphase, color="blue", linewidth=2.0, label=r"Coherent In-Phase: $\rho(\Delta f, \phi=0)$")
ax.plot(norm_spacing, rho_quad, color="orange", linestyle="--", linewidth=1.5, label=r"Coherent Quadrature: $\rho(\Delta f, \phi=\pi/2)$")
ax.plot(norm_spacing, rho_envelope, color="red", linewidth=2.2, label=r"Non-Coherent Envelope: $\sqrt{\rho_I^2 + \rho_Q^2}$")

ax.axvline(0.5, color="green", linestyle=":", linewidth=1.8, label=r"Coherent Min Spacing $\Delta f = 1/(2T_b)$ (500 Hz)")
ax.axvline(1.0, color="purple", linestyle=":", linewidth=1.8, label=r"Non-Coherent Min Spacing $\Delta f = 1/T_b$ (1000 Hz)")
ax.axhline(0.0, color="black", linestyle="-", linewidth=0.8)

ax.set_title("BFSK Cross-Correlation vs Normalized Tone Spacing (Orthogonality Condition)", fontsize=13)
ax.set_xlabel(r"Normalized Frequency Separation $\Delta f \cdot T_b = \Delta f / R_b$", fontsize=12)
ax.set_ylabel(r"Basis Cross-Correlation Coefficient $\rho$", fontsize=12)
ax.set_xlim(0, 3.0); ax.set_ylim(-0.35, 1.05); ax.grid(True)
ax.legend(loc="upper right", framealpha=0.95)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "5_bfsk_tone_spacing_orthogonality.png")
plt.show()


---
## 4. Coherent Correlator Detection & Decision Statistics

- **Coherent ASK:** Correlates received signal with $\phi(t) = \sqrt{2/T_b}\cos(2\pi f_c t)$ and applies threshold $\gamma = \sqrt{E_b/2} = \frac{A}{2}\sqrt{T_b/2}$.
- **Coherent BFSK:** Correlates with $\phi_1(t)$ and $\phi_2(t)$, forming differential decision statistic $D = Z_1 - Z_2$, sliced at threshold $\gamma = 0$.


In [ ]:
# Cell 6: Coherent Correlator Demodulation [Figure 3]
def add_awgn(sig: np.ndarray, snr_db: float, seed: Optional[int] = None):
    p_sig = np.mean(sig ** 2)
    snr_lin = 10.0 ** (snr_db / 10.0)
    sigma = np.sqrt(p_sig / snr_lin)
    rng = np.random.default_rng(seed)
    noise = rng.normal(0.0, sigma, size=len(sig))
    return sig + noise, sigma ** 2

def demod_ask_coh(r_t: np.ndarray, n_bits: int, tb: float = TB, fc: float = FC_ASK, fs: float = FS, amp: float = AMP):
    sps = int(fs * tb); dt = 1.0 / fs
    basis = np.sqrt(2.0 / tb) * np.cos(2.0 * np.pi * fc * (np.arange(sps) / fs))
    z = np.zeros(n_bits)
    for i in range(n_bits):
        z[i] = np.sum(r_t[i*sps:(i+1)*sps] * basis) * dt
    thresh = amp * np.sqrt(tb / 2.0) / 2.0
    return (z > thresh).astype(int), z, thresh

def demod_bfsk_coh(r_t: np.ndarray, n_bits: int, tb: float = TB, f1: float = F1_FSK, f2: float = F2_FSK, fs: float = FS):
    sps = int(fs * tb); dt = 1.0 / fs; t_bit = np.arange(sps) / fs
    b1 = np.sqrt(2.0 / tb) * np.cos(2.0 * np.pi * f1 * t_bit)
    b2 = np.sqrt(2.0 / tb) * np.cos(2.0 * np.pi * f2 * t_bit)
    z1 = np.zeros(n_bits); z2 = np.zeros(n_bits)
    for i in range(n_bits):
        r_seg = r_t[i*sps:(i+1)*sps]
        z1[i] = np.sum(r_seg * b1) * dt
        z2[i] = np.sum(r_seg * b2) * dt
    d = z1 - z2
    return (d > 0.0).astype(int), z1, z2, d

# Evaluate on 15 bits at 12 dB SNR
bits_15 = np.array([1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0])
_, _, ask_c = modulate_ask(bits_15, TB, FC_ASK, FS, AMP)
_, _, bfsk_c = modulate_bfsk(bits_15, TB, F1_FSK, F2_FSK, FS, AMP)
ask_n, _ = add_awgn(ask_c, 12.0, seed=77)
bfsk_n, _ = add_awgn(bfsk_c, 12.0, seed=78)
det_ask, z_ask, thresh_ask = demod_ask_coh(ask_n, len(bits_15), TB, FC_ASK, FS, AMP)
det_bfsk, _, _, d_bfsk = demod_bfsk_coh(bfsk_n, len(bits_15), TB, F1_FSK, F2_FSK, FS)

print(f"Original bits:    {list(bits_15)}")
print(f"Detected ASK:     {list(det_ask)} | Errors: {np.sum(det_ask != bits_15)}")
print(f"Detected BFSK:    {list(det_bfsk)} | Errors: {np.sum(det_bfsk != bits_15)}")

# Plot Figure 3: Correlator Outputs and Decision Thresholds
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
bit_indices = np.arange(len(bits_15)) + 0.5

axes[0].step(np.arange(len(bits_15)), bits_15, where="post", color="navy", linewidth=2.0, label="Transmitted Bits")
axes[0].set_ylabel("Data Bit"); axes[0].set_ylim(-0.2, 1.2)
axes[0].set_title("Transmitted Binary Sequence (SNR = 12 dB)")
axes[0].grid(True); axes[0].legend(loc="upper right")

axes[1].stem(bit_indices, z_ask, linefmt="crimson", markerfmt="ro", basefmt="k-", label=r"Correlator Output $Z$")
axes[1].axhline(thresh_ask, color="black", linestyle="--", linewidth=1.8, label=f"Threshold $\\gamma = {thresh_ask:.4f}$")
axes[1].set_ylabel("ASK Correlator"); axes[1].set_title("Coherent ASK Correlator Decision Statistic")
axes[1].grid(True); axes[1].legend(loc="upper right")

axes[2].stem(bit_indices, d_bfsk, linefmt="forestgreen", markerfmt="go", basefmt="k-", label=r"Differential Metric $D = Z_1 - Z_2$")
axes[2].axhline(0.0, color="black", linestyle="--", linewidth=1.8, label="Zero Decision Threshold")
axes[2].set_ylabel("BFSK Decision Metric"); axes[2].set_xlabel("Bit Index")
axes[2].set_title(r"Coherent BFSK Dual-Correlator Difference ($Z_1 - Z_2$)")
axes[2].grid(True); axes[2].legend(loc="upper right")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "3_correlator_outputs_and_decision.png")
plt.show()


---
### 4.2 Decision Statistic Probability Density Distributions

To verify the optimality of the decision thresholds, we observe the empirical probability density functions (histograms) of the decision statistics for binary `'0'` and `'1'` under AWGN at $E_b/N_0 = 8\text{ dB}$.


In [ ]:
# Cell 7: Decision Statistic Histograms under AWGN [Figure 4]
n_hist_bits = 5000
snr_hist_db = 8.0
hist_bits = generate_random_bits(n_hist_bits, seed=888)
_, _, ask_clean_hist = modulate_ask(hist_bits, TB, FC_ASK, FS, AMP)
_, _, bfsk_clean_hist = modulate_bfsk(hist_bits, TB, F1_FSK, F2_FSK, FS, AMP)

ask_noisy_hist, _ = add_awgn(ask_clean_hist, snr_hist_db, seed=991)
bfsk_noisy_hist, _ = add_awgn(bfsk_clean_hist, snr_hist_db, seed=992)

_, ask_corr_hist, ask_thresh_hist = demod_ask_coh(ask_noisy_hist, n_hist_bits, TB, FC_ASK, FS, AMP)
_, _, _, bfsk_stat_hist = demod_bfsk_coh(bfsk_noisy_hist, n_hist_bits, TB, F1_FSK, F2_FSK, FS)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

ax1.hist(ask_corr_hist[hist_bits == 0], bins=50, density=True, alpha=0.6, color="blue", label=r"Bit '0' ($s_0=0$)")
ax1.hist(ask_corr_hist[hist_bits == 1], bins=50, density=True, alpha=0.6, color="red", label=r"Bit '1' ($s_1=A\cos(\omega_c t)$)")
ax1.axvline(ask_thresh_hist, color="black", linestyle="--", linewidth=2.0, label=f"Optimum Threshold $\\gamma = {ask_thresh_hist:.4f}$")
ax1.set_title(f"Coherent ASK Correlator PDF ($E_b/N_0 = {snr_hist_db}$ dB)")
ax1.set_xlabel(r"Correlator Output $Z$"); ax1.set_ylabel("Probability Density")
ax1.grid(True); ax1.legend(loc="upper right")

ax2.hist(bfsk_stat_hist[hist_bits == 0], bins=50, density=True, alpha=0.6, color="purple", label=r"Bit '0' ($f_2=12$ kHz)")
ax2.hist(bfsk_stat_hist[hist_bits == 1], bins=50, density=True, alpha=0.6, color="green", label=r"Bit '1' ($f_1=10$ kHz)")
ax2.axvline(0.0, color="black", linestyle="--", linewidth=2.0, label=r"Optimum Threshold $\gamma = 0$")
ax2.set_title(f"Coherent BFSK Differential Metric PDF ($E_b/N_0 = {snr_hist_db}$ dB)")
ax2.set_xlabel(r"Decision Metric $D = Z_1 - Z_2$"); ax2.set_ylabel("Probability Density")
ax2.grid(True); ax2.legend(loc="upper right")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "4_decision_statistic_histograms.png")
plt.show()


---
## 5. Non-Coherent Detection: Envelope & Diode-LPF Processing

In non-coherent detection, the receiver operates directly on the signal envelope:
$$R(t) = \sqrt{s^2(t) + \hat{s}^2(t)}$$
followed by low-pass filtering and threshold slicing at $A/2 = 0.5\text{ V}$.


In [ ]:
# Cell 8: Non-Coherent Envelope Extraction Demonstration [Figure 7]
t_bits = np.array([1, 0, 1, 1, 0, 1, 0, 0, 1, 0])
t, bb, ask_c = modulate_ask(t_bits, TB, FC_ASK, FS, AMP)
ask_n, _ = add_awgn(ask_c, snr_db=14.0, seed=123)

analytic = signal.hilbert(ask_n)
env = np.abs(analytic)
b_lpf, a_lpf = signal.butter(4, 2.0 * RB / FS, btype="low")
smooth_env = signal.filtfilt(b_lpf, a_lpf, env)
rec_bits = (smooth_env > (AMP / 2.0)).astype(int)

fig, axes = plt.subplots(3, 1, figsize=(13, 7.5), sharex=True)
axes[0].plot(t * 1000, ask_n, color="crimson", linewidth=1.0)
axes[0].set_ylabel("Noisy ASK"); axes[0].set_title("Received Passband ASK Waveform (14 dB SNR)"); axes[0].grid(True)

axes[1].plot(t * 1000, env, color="gray", alpha=0.4, label="Raw Rectified Envelope")
axes[1].plot(t * 1000, smooth_env, color="darkorange", linewidth=2.0, label="Post-LPF Smoothed Envelope")
axes[1].axhline(AMP / 2.0, color="black", linestyle="--", linewidth=1.5, label="Threshold (0.5 V)")
axes[1].set_ylabel("Envelope (V)"); axes[1].set_title("Non-Coherent Envelope Extraction"); axes[1].grid(True); axes[1].legend(loc="upper right")

axes[2].step(t * 1000, bb, where="post", color="navy", linewidth=2.0, label="Transmitted Bits")
axes[2].step(t * 1000, rec_bits, where="post", color="red", linestyle="--", linewidth=1.5, label="Decided Bits")
axes[2].set_ylabel("Data Bits"); axes[2].set_xlabel("Time (milliseconds)"); axes[2].set_title("Data Bit Comparison"); axes[2].grid(True); axes[2].legend(loc="upper right")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "7_noncoherent_envelope_detection.png")
plt.show()


---
## 6. Monte Carlo Bit Error Rate (BER) Simulation vs Theory

We evaluate Bit Error Rate over $E_b/N_0 \in [0, 14]\text{ dB}$ across four schemes:
1. **Coherent ASK:** $P_b = Q(\sqrt{E_b/N_0})$
2. **Coherent BFSK:** $P_b = Q(\sqrt{E_b/N_0})$
3. **Non-Coherent BFSK:** $P_b = \frac{1}{2} \exp\left(-\frac{E_b}{2 N_0}\right)$
4. **Non-Coherent ASK:** $P_b \approx \frac{1}{2} \exp\left(-\frac{E_b}{4 N_0}\right)$


In [ ]:
# Cell 9: Fast Vectorized Monte Carlo BER Simulation [Figure 6]
def simulate_ber_sweep(eb_no_db_range: np.ndarray, n_bits: int = 50000):
    bits = generate_random_bits(n_bits, seed=123)
    ber_ask_coh = np.zeros(len(eb_no_db_range))
    ber_ask_noncoh = np.zeros(len(eb_no_db_range))
    ber_bfsk_coh = np.zeros(len(eb_no_db_range))
    ber_bfsk_noncoh = np.zeros(len(eb_no_db_range))

    for idx, eb_no_db in enumerate(eb_no_db_range):
        eb_no_lin = 10.0 ** (eb_no_db / 10.0)
        n0 = 1.0 / eb_no_lin
        sigma = np.sqrt(n0 / 2.0)
        rng = np.random.default_rng(2000 + idx)

        # Coherent ASK
        s_ask = np.where(bits == 1, np.sqrt(2.0), 0.0)
        r_ask = s_ask + rng.normal(0.0, sigma, size=n_bits)
        ber_ask_coh[idx] = np.mean((r_ask > (np.sqrt(2.0) / 2.0)).astype(int) != bits)

        # Non-coherent ASK
        r_ask_env = np.sqrt(r_ask ** 2 + rng.normal(0.0, sigma, size=n_bits) ** 2)
        ber_ask_noncoh[idx] = np.mean((r_ask_env > (np.sqrt(2.0) / 2.0)).astype(int) != bits)

        # Coherent BFSK
        s_fsk_1 = np.where(bits == 1, 1.0, 0.0); s_fsk_2 = np.where(bits == 0, 1.0, 0.0)
        z1 = s_fsk_1 + rng.normal(0.0, sigma, size=n_bits)
        z2 = s_fsk_2 + rng.normal(0.0, sigma, size=n_bits)
        ber_bfsk_coh[idx] = np.mean((z1 > z2).astype(int) != bits)

        # Non-coherent BFSK
        r1_env = np.sqrt(z1 ** 2 + rng.normal(0.0, sigma, size=n_bits) ** 2)
        r2_env = np.sqrt(z2 ** 2 + rng.normal(0.0, sigma, size=n_bits) ** 2)
        ber_bfsk_noncoh[idx] = np.mean((r1_env > r2_env).astype(int) != bits)

    return {
        "eb_no_db": eb_no_db_range,
        "ask_coh": ber_ask_coh,
        "ask_noncoh": ber_ask_noncoh,
        "bfsk_coh": ber_bfsk_coh,
        "bfsk_noncoh": ber_bfsk_noncoh
    }

eb_range = np.linspace(0.0, 14.0, 15)
sim = simulate_ber_sweep(eb_range, n_bits=50000)
eb_fine = np.linspace(0.0, 14.0, 100)
eb_lin = 10.0 ** (eb_fine / 10.0)

th_coh = 0.5 * sp.erfc(np.sqrt(eb_lin / 2.0))
th_bfsk_noncoh = 0.5 * np.exp(-eb_lin / 2.0)
th_ask_noncoh = 0.5 * np.exp(-eb_lin / 4.0)

plt.figure(figsize=(11, 7.5))
plt.semilogy(eb_fine, th_coh, "r-", linewidth=1.8, label="Coherent ASK/BFSK (Theory)")
plt.semilogy(sim["eb_no_db"], sim["ask_coh"], "ro", markersize=6, fillstyle="none", markeredgewidth=1.8, label="Coherent ASK (Sim)")
plt.semilogy(sim["eb_no_db"], sim["bfsk_coh"], "bs", markersize=6, fillstyle="none", markeredgewidth=1.8, label="Coherent BFSK (Sim)")
plt.semilogy(eb_fine, th_bfsk_noncoh, "g-", linewidth=1.8, label="Non-Coherent BFSK (Theory)")
plt.semilogy(sim["eb_no_db"], sim["bfsk_noncoh"], "g^", markersize=6, fillstyle="none", markeredgewidth=1.8, label="Non-Coherent BFSK (Sim)")
plt.semilogy(eb_fine, th_ask_noncoh, "m--", linewidth=1.5, label="Non-Coherent ASK (Bound)")
plt.semilogy(sim["eb_no_db"], sim["ask_noncoh"], "md", markersize=6, fillstyle="none", markeredgewidth=1.5, label="Non-Coherent ASK (Sim)")

plt.title("Bit Error Rate (BER) Comparison over AWGN Channel", fontsize=13)
plt.xlabel(r"Energy per Bit to Noise Spectral Density $E_b/N_0$ (dB)", fontsize=12)
plt.ylabel("Bit Error Rate (BER)", fontsize=12)
plt.xlim(0, 14); plt.ylim(1e-5, 0.6); plt.grid(True, which="both", linestyle="--", alpha=0.6)
plt.legend(loc="lower left", framealpha=0.95)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "6_ber_performance_comparison.png")
plt.show()


---
## 7. Conclusions & Laboratory Findings

1. **Orthogonality Conditions:**
   - Coherent BFSK achieves exact orthogonality at minimum tone spacing $\Delta f_{\min} = 1/(2T_b) = 500\text{ Hz}$.
   - Non-coherent BFSK requires $\Delta f_{\min} = 1/T_b = 1000\text{ Hz}$ to guarantee zero cross-talk across all unknown carrier phases.
2. **BER Performance:**
   - Coherent ASK and Coherent BFSK exhibit identical BER performance $P_b = Q(\sqrt{E_b/N_0})$ for equal average bit energy.
   - Non-coherent BFSK incurs only a modest $\approx 1\text{ dB}$ power penalty relative to coherent BFSK at $10^{-4}$ BER while completely eliminating receiver carrier tracking complexity.
3. **Spectral Containment:**
   - Continuous Phase FSK (CPFSK) eliminates phase discontinuities, suppressing out-of-band spectral spillover.
4. **All 8 Publication Figures:**
   - All figures generated and saved to `outputs/` in 300 DPI publication quality.
